In [ ]:

import sys, subprocess
try:
    import scipy  # noqa: F401
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scipy", "-q"])

# ============================================================
# Q3 ATTENTION BENCHMARK — UPGRADED
# ------------------------------------------------------------
# Research question:
# Can the model shift focus between sub-tasks in a complex,
# multi-part prompt without losing track of the main task?
#
# Upgrades in this version:
# - explicit subtask-burden levels (single / optional light / full)
# - optional single-long control (length-matched instruction control)
# - optional EPU-middle condition
# - stronger structural null handling for auxiliary labels
# - structured execution / hard-failure diagnostics
# - failure-mode classification
# - evidence-grounding and confidence diagnostics
# - subtask dependency analysis
# - partial saves after each condition
# ============================================================

import json
import math
import re
import hashlib
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
from scipy.stats import binomtest
from tqdm.auto import tqdm
from IPython.display import display

try:
    import kaggle_benchmarks as kbench
except Exception as e:
    raise RuntimeError("This notebook must run in Kaggle with kaggle_benchmarks available.") from e

# ---------------------------
# CONFIG
# ---------------------------
INPUT_CSV = "/kaggle/input/datasets/bigdataanalysis1/epu-800/EPU_benchmark_800_balanced.csv"

SEED = 42
N_ROWS = 600
MAX_CHARS = 5000
BOOT_N = 4000

MODEL_NAMES = [
    # "google/gemini-2.5-flash",
    # "google/gemini-2.5-pro",
    # "anthropic/claude-sonnet-4-6@default",
    # "deepseek-ai/deepseek-v3.2",
    "qwen/qwen3-235b-a22b-instruct-2507",
]

# Keep defaults near current cost. Increase only if you want stronger scientific coverage.
ACTIVE_BUNDLES = ["full"]                 # options: ["light"], ["full"], ["light","full"]
INCLUDE_SINGLE_LONG_CONTROL = True        # length-matched prompt control
INCLUDE_EPU_MIDDLE = False                # add order=middle
SAVE_ROW_CHECKPOINT_EVERY = 50

OUT_DIR = Path("/kaggle/working/epu_attention_q3_upgraded_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MISSING_SENTINEL = -1
PRIMARY_BUNDLE = ACTIVE_BUNDLES[-1] if ACTIVE_BUNDLES else "full"

ALL_AUX_FIELDS = ["who", "actions", "effects", "mention_foreign", "mainly_foreign"]
BUNDLE_FIELDS = {
    "light": ["who", "actions"],
    "full": ["who", "actions", "effects", "mention_foreign", "mainly_foreign"],
}

ORDER_SEQUENCE = ["epu_first", "epu_last"] if not INCLUDE_EPU_MIDDLE else ["epu_first", "epu_middle", "epu_last"]

# ---------------------------
# LOAD + CLEAN
# ---------------------------
df = pd.read_csv(INPUT_CSV)

required_cols = [
    "article_key",
    "content",
    "gold_epu",
    "gold_who",
    "gold_actions",
    "gold_effects",
    "mention_foreign",
    "mainly_foreign",
    "disagreement_flag",
    "newspaper",
    "year",
    "content_len",
]
for c in required_cols:
    if c not in df.columns:
        df[c] = np.nan

mask = (
    df["content"].notna()
    & (df["content"].astype(str).str.len() > 300)
    & df["gold_epu"].notna()
)
clean = df.loc[mask].copy()

clean["article_key"] = clean["article_key"].astype(str)
bad_key_mask = clean["article_key"].isin(["", "nan", "None"])
if bad_key_mask.any():
    clean.loc[bad_key_mask, "article_key"] = [f"generated_key_{i}" for i in range(int(bad_key_mask.sum()))]

clean["gold_epu"] = clean["gold_epu"].astype(int)

def derive_binary_gold(frame: pd.DataFrame, preferred_gold_col: str, fallback_raw_col: str) -> pd.Series:
    if preferred_gold_col in frame.columns and frame[preferred_gold_col].notna().any():
        return frame[preferred_gold_col].fillna(0).astype(float).ge(0.5).astype(int)
    return frame[fallback_raw_col].fillna(0).astype(float).ge(0.5).astype(int)

clean["gold_mention_foreign"] = derive_binary_gold(clean, "gold_mention_foreign", "mention_foreign")
clean["gold_mainly_foreign"] = derive_binary_gold(clean, "gold_mainly_foreign", "mainly_foreign")

for c in ["gold_who", "gold_actions", "gold_effects"]:
    clean[c] = clean[c].fillna(MISSING_SENTINEL).astype(int)

clean["disagreement_flag"] = clean["disagreement_flag"].fillna(0).astype(int)
clean["year"] = pd.to_numeric(clean["year"], errors="coerce")
clean["content_len"] = pd.to_numeric(clean["content_len"], errors="coerce")
clean = clean.drop_duplicates(subset=["article_key"]).reset_index(drop=True)

print(f"Loaded rows: {len(df):,}")
print(f"Clean usable rows: {len(clean):,}")
display(clean["gold_epu"].value_counts(dropna=False).sort_index())
display(clean["disagreement_flag"].value_counts(dropna=False).sort_index())

# ---------------------------
# SAMPLING
# ---------------------------
def smart_truncate(text: str, max_chars: int = 5000) -> str:
    text = str(text)
    if len(text) <= max_chars:
        return text
    head = max_chars // 2
    tail = max_chars - head - 32
    return text[:head] + "\n\n[...TRUNCATED...]\n\n" + text[-tail:]

def make_sample_frame(data: pd.DataFrame, n: int, seed: int = 42) -> pd.DataFrame:
    # Primary balance by gold_epu, with soft balancing by disagreement flag.
    rng = np.random.default_rng(seed)
    pos = data[data["gold_epu"] == 1].copy()
    neg = data[data["gold_epu"] == 0].copy()

    if len(pos) == 0 or len(neg) == 0:
        out = data.sample(min(n, len(data)), random_state=seed).reset_index(drop=True)
    else:
        n_pos = min(len(pos), n // 2)
        n_neg = min(len(neg), n - n_pos)
        pos_s = pos.sample(n=n_pos, random_state=seed)
        neg_s = neg.sample(n=n_neg, random_state=seed + 1)
        out = pd.concat([pos_s, neg_s], axis=0).sample(frac=1.0, random_state=seed + 2).reset_index(drop=True)

    out["content_for_eval"] = out["content"].astype(str).apply(lambda x: smart_truncate(x, MAX_CHARS))
    out["orig_content_len"] = out["content"].astype(str).str.len()
    out["eval_content_len"] = out["content_for_eval"].astype(str).str.len()
    out["was_truncated"] = (out["orig_content_len"] > out["eval_content_len"]).astype(int)
    out["article_key"] = out["article_key"].astype(str)
    return out

pilot = make_sample_frame(clean, N_ROWS, SEED).copy()
pilot_path = OUT_DIR / "pilot_rows_q3_upgraded.csv"
pilot.to_csv(pilot_path, index=False)

# ---------------------------
# HELPERS
# ---------------------------
def maybe_int(x):
    if x is None:
        return None
    try:
        if pd.isna(x):
            return None
    except Exception:
        pass
    try:
        return int(x)
    except Exception:
        return None

def safe_float(x, default=np.nan):
    try:
        return float(x)
    except Exception:
        return default

def binary_match(pred, gold, missing_sentinel=MISSING_SENTINEL):
    gold_i = maybe_int(gold)
    pred_i = maybe_int(pred)
    if gold_i is None or gold_i == missing_sentinel:
        return None
    if pred_i is None:
        return 0.0
    return 1.0 if pred_i == gold_i else 0.0

def default_out():
    return {
        "epu": None,
        "who": None,
        "actions": None,
        "effects": None,
        "mention_foreign": None,
        "mainly_foreign": None,
        "confidence": 0.0,
        "focus_excerpt": "",
        "raw_text": "",
        "json_valid": 0,
        "parse_status": "invalid",
    }

def parse_llm_json(raw_text):
    out = default_out()
    out["raw_text"] = "" if raw_text is None else str(raw_text)[:5000]

    if raw_text is None:
        out["parse_status"] = "none"
        return out

    text = str(raw_text).strip()
    text = text.replace("```json", "").replace("```", "").strip()
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        text = match.group(0)

    try:
        obj = json.loads(text)
        if not isinstance(obj, dict):
            out["parse_status"] = "not_dict"
            return out
        out["epu"] = maybe_int(obj.get("epu"))
        out["who"] = maybe_int(obj.get("who"))
        out["actions"] = maybe_int(obj.get("actions"))
        out["effects"] = maybe_int(obj.get("effects"))
        out["mention_foreign"] = maybe_int(obj.get("mention_foreign"))
        out["mainly_foreign"] = maybe_int(obj.get("mainly_foreign"))
        out["confidence"] = max(0.0, min(1.0, safe_float(obj.get("confidence"), 0.0)))
        out["focus_excerpt"] = str(obj.get("focus_excerpt", ""))[:500]
        out["json_valid"] = 1
        out["parse_status"] = "ok"
        return out
    except Exception:
        out["parse_status"] = "json_error"
        return out

def model_json_call(llm, prompt: str):
    try:
        raw = llm.prompt(prompt)
        return parse_llm_json(raw)
    except Exception as e:
        out = default_out()
        out["raw_text"] = f"ERROR: {str(e)[:5000]}"
        out["parse_status"] = "call_error"
        return out

def excerpt_in_target(excerpt: str, article_text: str) -> int:
    excerpt = str(excerpt or "").strip()
    article_text = str(article_text or "")
    if len(excerpt) < 10:
        return 0
    return int(excerpt in article_text)

# ---------------------------
# PROMPT BUILDERS
# ---------------------------
CONTROL_PADDING = """
Administrative control text:
- Keep track of the decision rule internally.
- Check the requested output fields before finishing.
- Ensure the short quote comes from inside the ARTICLE only.
- Do not use outside knowledge.
- Ignore document formatting and focus on article meaning.
- Maintain consistency between the label and the evidence.
"""

def make_control_padding(multiplier: int = 5) -> str:
    return "\n".join([CONTROL_PADDING.strip()] * multiplier)

def format_label_rules() -> str:
    return """Labeling rule:
- EPU = 1 only if the ARTICLE discusses uncertainty about government policy,
  policy decisions, possible policy actions, political outcomes affecting policy,
  or the economic effects of policy.
- EPU = 0 for general economics, elections, markets, war, uncertainty, or foreign events
  unless the uncertainty is specifically tied to economic policy.

Null handling:
- If epu = 0, then set "who", "actions", and "effects" to null.
- If a subtask is not requested in this condition, return null for it.
- Use only the ARTICLE below.
- focus_excerpt must be copied from the ARTICLE only.
"""

def build_single_prompt(article_text: str) -> str:
    return f"""You are evaluating the ARTICLE below.

Task:
Decide whether the ARTICLE expresses ECONOMIC POLICY UNCERTAINTY (EPU).

{format_label_rules()}

IMPORTANT:
- Return ONLY valid JSON.
- No markdown. No explanation. No code fence.

Return exactly these keys:
{{
  "epu": 0 or 1,
  "confidence": number between 0 and 1,
  "focus_excerpt": "short quote from the article only"
}}

ARTICLE START
{article_text}
ARTICLE END""".strip()

def build_single_long_control_prompt(article_text: str) -> str:
    return f"""You are evaluating the ARTICLE below.

Task:
Decide whether the ARTICLE expresses ECONOMIC POLICY UNCERTAINTY (EPU).

{format_label_rules()}

Before answering, silently perform this administrative checklist:
{make_control_padding(multiplier=6)}

IMPORTANT:
- Return ONLY valid JSON.
- No markdown. No explanation. No code fence.

Return exactly these keys:
{{
  "epu": 0 or 1,
  "confidence": number between 0 and 1,
  "focus_excerpt": "short quote from the article only"
}}

ARTICLE START
{article_text}
ARTICLE END""".strip()

def bundle_instructions(bundle_name: str):
    fields = BUNDLE_FIELDS[bundle_name]
    items = []
    if "who" in fields:
        items.append('Code whether uncertainty is about who.')
    if "actions" in fields:
        items.append('Code whether uncertainty is about actions.')
    if "effects" in fields:
        items.append('Code whether uncertainty is about effects.')
    if "mention_foreign" in fields:
        items.append('Code mention_foreign.')
    if "mainly_foreign" in fields:
        items.append('Code mainly_foreign.')
    return items, fields

def build_multi_prompt(article_text: str, bundle_name: str = "full", order: str = "epu_first") -> str:
    items, requested_fields = bundle_instructions(bundle_name)
    aux_lines = [f"{i+1}. {txt}" for i, txt in enumerate(items)]

    if order == "epu_first":
        ordered = ['1. Decide whether the ARTICLE expresses ECONOMIC POLICY UNCERTAINTY (EPU).']
        start_idx = 2
        ordered += [f"{start_idx + i}. {txt}" for i, txt in enumerate(items)]
        ordered.append(f"{start_idx + len(items)}. Provide a short supporting quote from the ARTICLE only.")
    elif order == "epu_middle":
        split = max(1, len(items) // 2)
        first_part = items[:split]
        second_part = items[split:]
        ordered = [f"{i+1}. {txt}" for i, txt in enumerate(first_part)]
        ordered.append(f"{len(first_part)+1}. Decide whether the ARTICLE expresses ECONOMIC POLICY UNCERTAINTY (EPU).")
        base = len(first_part) + 2
        ordered += [f"{base + i}. {txt}" for i, txt in enumerate(second_part)]
        ordered.append(f"{base + len(second_part)}. Provide a short supporting quote from the ARTICLE only.")
    elif order == "epu_last":
        ordered = [f"{i+1}. {txt}" for i, txt in enumerate(items)]
        ordered.append(f"{len(items)+1}. Provide a short supporting quote from the ARTICLE only.")
        ordered.append(f"{len(items)+2}. Only after finishing the above, decide whether the ARTICLE expresses ECONOMIC POLICY UNCERTAINTY (EPU).")
    else:
        raise ValueError(f"Unknown order: {order}")

    return f"""You are evaluating the ARTICLE below.

Perform ALL subtasks below in order:
{chr(10).join(ordered)}

{format_label_rules()}

IMPORTANT:
- Return ONLY valid JSON.
- No markdown. No explanation. No code fence.
- For auxiliary fields not requested in this condition, return null.

Return exactly these keys:
{{
  "epu": 0 or 1,
  "who": 0 or 1 or null,
  "actions": 0 or 1 or null,
  "effects": 0 or 1 or null,
  "mention_foreign": 0 or 1 or null,
  "mainly_foreign": 0 or 1 or null,
  "confidence": number between 0 and 1,
  "focus_excerpt": "short quote from the article only"
}}

ARTICLE START
{article_text}
ARTICLE END""".strip()

def active_conditions():
    conds = [{"condition": "single_focus", "kind": "single", "bundle": None, "order": None, "requested_fields": []}]
    if INCLUDE_SINGLE_LONG_CONTROL:
        conds.append({"condition": "single_long_control", "kind": "single_control", "bundle": None, "order": None, "requested_fields": []})
    for bundle in ACTIVE_BUNDLES:
        _, fields = bundle_instructions(bundle)
        for order in ORDER_SEQUENCE:
            conds.append({
                "condition": f"{bundle}_focus_{order}",
                "kind": "multi",
                "bundle": bundle,
                "order": order,
                "requested_fields": fields.copy(),
            })
    return conds

CONDITIONS = active_conditions()

# ---------------------------
# STATISTICS
# ---------------------------
def wilson_ci(successes: int, n: int, z: float = 1.959963984540054):
    if n <= 0:
        return (np.nan, np.nan)
    phat = successes / n
    denom = 1.0 + (z**2) / n
    center = (phat + (z**2) / (2 * n)) / denom
    margin = (z / denom) * math.sqrt((phat * (1 - phat) / n) + (z**2) / (4 * n**2))
    return (max(0.0, center - margin), min(1.0, center + margin))

def paired_bootstrap_diff_ci(a_values, b_values, n_boot=4000, seed=42, alpha=0.05):
    a = np.asarray(a_values, dtype=float)
    b = np.asarray(b_values, dtype=float)
    mask = ~(np.isnan(a) | np.isnan(b))
    a = a[mask]
    b = b[mask]
    n = len(a)
    if n == 0:
        return (np.nan, np.nan)
    diffs = a - b
    rng = np.random.default_rng(seed)
    boot = np.empty(n_boot, dtype=float)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot[i] = diffs[idx].mean()
    return (float(np.quantile(boot, alpha / 2)), float(np.quantile(boot, 1 - alpha / 2)))

def exact_mcnemar(a_correct, b_correct):
    x = np.asarray(a_correct, dtype=float)
    y = np.asarray(b_correct, dtype=float)
    mask = ~(np.isnan(x) | np.isnan(y))
    x = x[mask].astype(int)
    y = y[mask].astype(int)
    b = int(np.sum((x == 1) & (y == 0)))
    c = int(np.sum((x == 0) & (y == 1)))
    n = b + c
    if n == 0:
        return {
            "b_hurt": b,
            "c_helped": c,
            "discordant_n": n,
            "mcnemar_p": 1.0,
            "hurt_help_ratio": np.nan,
        }
    pval = binomtest(min(b, c), n=n, p=0.5, alternative="two-sided").pvalue
    ratio = np.inf if c == 0 and b > 0 else (b / c if c > 0 else np.nan)
    return {
        "b_hurt": b,
        "c_helped": c,
        "discordant_n": n,
        "mcnemar_p": float(pval),
        "hurt_help_ratio": ratio,
    }

def summarize_pair(a_correct, b_correct, a_pred, b_pred, compare_name):
    a = np.asarray(a_correct, dtype=float)
    b = np.asarray(b_correct, dtype=float)
    pred_a = np.asarray(a_pred, dtype=float)
    pred_b = np.asarray(b_pred, dtype=float)

    mask = ~(np.isnan(a) | np.isnan(b))
    a = a[mask]
    b = b[mask]
    pred_a = pred_a[mask]
    pred_b = pred_b[mask]

    n = len(a)
    acc_a = float(np.mean(a)) if n else np.nan
    acc_b = float(np.mean(b)) if n else np.nan
    delta_pp = acc_b - acc_a if n else np.nan
    ci_lo, ci_hi = paired_bootstrap_diff_ci(b, a, n_boot=BOOT_N, seed=SEED)

    flip_mask = ~(np.isnan(pred_a) | np.isnan(pred_b))
    flip_n = int(np.sum(flip_mask))
    flip_k = int(np.sum(pred_a[flip_mask] != pred_b[flip_mask])) if flip_n else 0
    flip_rate = flip_k / flip_n if flip_n else np.nan
    flip_lo, flip_hi = wilson_ci(flip_k, flip_n)

    hurt_k = int(np.sum((a == 1) & (b == 0))) if n else 0
    help_k = int(np.sum((a == 0) & (b == 1))) if n else 0
    mc = exact_mcnemar(a, b)

    return {
        "comparison": compare_name,
        "n_rows": n,
        "acc_a": acc_a,
        "acc_b": acc_b,
        "delta_pp": delta_pp,
        "delta_ci_low": ci_lo,
        "delta_ci_high": ci_hi,
        "flip_rate": flip_rate,
        "flip_rate_ci_low": flip_lo,
        "flip_rate_ci_high": flip_hi,
        "hurt_rate": hurt_k / n if n else np.nan,
        "help_rate": help_k / n if n else np.nan,
        **mc,
    }

def target_condition_name(bundle: str, order: str) -> str:
    return f"{bundle}_focus_{order}"

def classify_q3_failure(rec: dict) -> str:
    first_name = target_condition_name(PRIMARY_BUNDLE, "epu_first")
    last_name = target_condition_name(PRIMARY_BUNDLE, "epu_last")
    first_acc = rec.get(f"{first_name}_acc", np.nan)
    last_acc = rec.get(f"{last_name}_acc", np.nan)
    single_acc = rec.get("single_focus_acc", np.nan)

    first_json = rec.get(f"{first_name}_json_valid_rate", np.nan)
    last_json = rec.get(f"{last_name}_json_valid_rate", np.nan)
    first_comp = rec.get(f"{first_name}_completion_share_mean", np.nan)
    last_comp = rec.get(f"{last_name}_completion_share_mean", np.nan)

    if (
        (pd.notna(first_json) and first_json < 0.90)
        or (pd.notna(last_json) and last_json < 0.90)
        or (pd.notna(first_comp) and first_comp < 0.70)
        or (pd.notna(last_comp) and last_comp < 0.70)
    ):
        return "structured_execution_failure"

    first_pos = rec.get(f"{first_name}_acc_gold1", np.nan)
    first_neg = rec.get(f"{first_name}_acc_gold0", np.nan)
    last_pos = rec.get(f"{last_name}_acc_gold1", np.nan)
    last_neg = rec.get(f"{last_name}_acc_gold0", np.nan)

    order_delta = last_acc - first_acc if pd.notna(last_acc) and pd.notna(first_acc) else np.nan
    burden_first = first_acc - single_acc if pd.notna(first_acc) and pd.notna(single_acc) else np.nan
    burden_last = last_acc - single_acc if pd.notna(last_acc) and pd.notna(single_acc) else np.nan

    if pd.notna(order_delta) and order_delta <= -0.02:
        if pd.notna(first_pos) and pd.notna(last_pos) and pd.notna(first_neg) and pd.notna(last_neg):
            if (last_pos < first_pos) and (last_neg > first_neg):
                return "anti_epu_order_shift"
            if (last_pos > first_pos) and (last_neg < first_neg):
                return "pro_epu_order_shift"
        return "order_sensitive_main_task_loss"

    if pd.notna(order_delta) and order_delta >= 0.02:
        if pd.notna(first_pos) and pd.notna(last_pos) and pd.notna(first_neg) and pd.notna(last_neg):
            if (last_pos > first_pos) and (last_neg < first_neg):
                return "pro_epu_order_shift"
            if (last_pos < first_pos) and (last_neg > first_neg):
                return "anti_epu_order_shift"
        return "order_sensitive_main_task_gain"

    if pd.notna(burden_first) and pd.notna(burden_last):
        if burden_first <= -0.02 and burden_last <= -0.02:
            return "multitask_burden_degradation"
        if abs(burden_first) <= 0.01 and abs(burden_last) <= 0.01 and abs(order_delta) <= 0.01:
            return "q3_robust"

    return "mixed"

# ---------------------------
# CONDITION EXECUTION
# ---------------------------
def build_prompt_for_condition(article_text: str, cond_meta: dict) -> str:
    kind = cond_meta["kind"]
    if kind == "single":
        return build_single_prompt(article_text)
    if kind == "single_control":
        return build_single_long_control_prompt(article_text)
    if kind == "multi":
        return build_multi_prompt(article_text, bundle_name=cond_meta["bundle"], order=cond_meta["order"])
    raise ValueError(f"Unknown condition kind: {kind}")

def requested_field_count(cond_meta: dict) -> int:
    return len(cond_meta["requested_fields"])

def slugify_model_name(model_name: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", model_name)

run_status_records = []

def run_condition_for_model(llm, model_name: str, cond_meta: dict, pilot_df: pd.DataFrame):
    cond = cond_meta["condition"]
    model_slug = slugify_model_name(model_name)
    out_path = OUT_DIR / f"partial_{model_slug}_{cond}.csv"

    rows = []
    n_req = requested_field_count(cond_meta)
    for i, row in enumerate(tqdm(pilot_df.to_dict(orient="records"), total=len(pilot_df), desc=f"{model_name}::{cond}"), start=1):
        article_text = row["content_for_eval"]
        out = model_json_call(llm, build_prompt_for_condition(article_text, cond_meta))

        rec = {
            "article_key": row["article_key"],
            "condition": cond,
            "llm_name": model_name,
            f"{cond}_epu_pred": maybe_int(out["epu"]),
            f"{cond}_epu_correct": binary_match(out["epu"], row["gold_epu"]),
            f"{cond}_confidence": safe_float(out["confidence"]),
            f"{cond}_focus_excerpt": out["focus_excerpt"],
            f"{cond}_excerpt_in_target": excerpt_in_target(out["focus_excerpt"], article_text),
            f"{cond}_json_valid": int(out["json_valid"]),
            f"{cond}_parse_status": out["parse_status"],
            f"{cond}_raw_json": out["raw_text"],
        }

        aux_scores = []
        present_requested = 0
        for field in ALL_AUX_FIELDS:
            gold = row.get(f"gold_{field}", MISSING_SENTINEL)
            pred = maybe_int(out[field])
            correct = binary_match(pred, gold)
            requested = int(field in cond_meta["requested_fields"])
            present = int(pred is not None)

            rec[f"{cond}_{field}_pred"] = pred
            rec[f"{cond}_{field}_correct"] = correct
            rec[f"{cond}_{field}_requested"] = requested
            rec[f"{cond}_{field}_present"] = present

            if requested:
                present_requested += present
                if correct is not None:
                    aux_scores.append(float(correct))

        rec[f"{cond}_requested_task_count"] = n_req
        rec[f"{cond}_task_completion_count"] = present_requested
        rec[f"{cond}_completion_share"] = (present_requested / n_req) if n_req > 0 else np.nan
        rec[f"{cond}_aux_mean"] = np.nan if len(aux_scores) == 0 else float(np.mean(aux_scores))
        rows.append(rec)

        if i % SAVE_ROW_CHECKPOINT_EVERY == 0:
            pd.DataFrame(rows).to_csv(out_path, index=False)

    out_df = pd.DataFrame(rows)
    out_df.to_csv(out_path, index=False)

    parse_fail_rate = 1.0 - out_df[f"{cond}_json_valid"].mean()
    status = {
        "llm_name": model_name,
        "condition": cond,
        "status": "finished",
        "n_rows": len(out_df),
        "json_valid_rate": out_df[f"{cond}_json_valid"].mean(),
        "parse_fail_rate": parse_fail_rate,
        "completion_share_mean": out_df[f"{cond}_completion_share"].mean() if n_req > 0 else np.nan,
        "partial_path": str(out_path),
    }
    run_status_records.append(status)
    pd.DataFrame(run_status_records).to_csv(OUT_DIR / "run_status_q3_upgraded.csv", index=False)
    return out_df

# ---------------------------
# RUN MODELS / CONDITIONS
# ---------------------------
records_by_model = []

for model_name in MODEL_NAMES:
    print(f"\nRunning {model_name}")
    llm = kbench.llms[model_name]

    base_df = pilot.copy()
    base_df["llm_name"] = model_name

    condition_outputs = []
    for cond_meta in CONDITIONS:
        cond_df = run_condition_for_model(llm, model_name, cond_meta, pilot)
        cond_small = cond_df.drop(columns=["condition", "llm_name"], errors="ignore")
        base_df = base_df.merge(cond_small, on="article_key", how="left")
        condition_outputs.append(cond_df)

    records_by_model.append(base_df)

wide_df = pd.concat(records_by_model, axis=0, ignore_index=True)
wide_path = OUT_DIR / "q3_results_wide_upgraded.csv"
wide_df.to_csv(wide_path, index=False)

# ---------------------------
# LONG VERSION
# ---------------------------
long_records = []
for row in wide_df.to_dict(orient="records"):
    common = {
        "llm_name": row["llm_name"],
        "article_key": row["article_key"],
        "gold_epu": row["gold_epu"],
        "disagreement_flag": row["disagreement_flag"],
        "newspaper": row["newspaper"],
        "year": row["year"],
        "content_len": row["content_len"],
        "orig_content_len": row["orig_content_len"],
        "eval_content_len": row["eval_content_len"],
        "was_truncated": row["was_truncated"],
    }
    for cond_meta in CONDITIONS:
        cond = cond_meta["condition"]
        rec = dict(common)
        rec["condition"] = cond
        rec["bundle"] = cond_meta["bundle"]
        rec["order"] = cond_meta["order"]
        rec["condition_kind"] = cond_meta["kind"]
        rec["requested_task_count"] = requested_field_count(cond_meta)
        rec["epu_pred"] = row.get(f"{cond}_epu_pred")
        rec["epu_correct"] = row.get(f"{cond}_epu_correct")
        rec["confidence"] = row.get(f"{cond}_confidence")
        rec["focus_excerpt"] = row.get(f"{cond}_focus_excerpt")
        rec["excerpt_in_target"] = row.get(f"{cond}_excerpt_in_target")
        rec["json_valid"] = row.get(f"{cond}_json_valid")
        rec["parse_status"] = row.get(f"{cond}_parse_status")
        rec["task_completion_count"] = row.get(f"{cond}_task_completion_count")
        rec["completion_share"] = row.get(f"{cond}_completion_share")
        rec["aux_mean"] = row.get(f"{cond}_aux_mean")
        for field in ALL_AUX_FIELDS:
            rec[f"{field}_pred"] = row.get(f"{cond}_{field}_pred")
            rec[f"{field}_correct"] = row.get(f"{cond}_{field}_correct")
            rec[f"{field}_requested"] = row.get(f"{cond}_{field}_requested")
            rec[f"{field}_present"] = row.get(f"{cond}_{field}_present")
        long_records.append(rec)

long_df = pd.DataFrame(long_records)
long_path = OUT_DIR / "q3_results_long_upgraded.csv"
long_df.to_csv(long_path, index=False)

# ---------------------------
# SUMMARIES
# ---------------------------
def summarize_condition(g: pd.DataFrame, cond: str):
    epu_vec = g[f"{cond}_epu_correct"].astype(float).to_numpy()
    n = int(np.sum(~np.isnan(epu_vec)))
    k = int(np.nansum(epu_vec)) if n else 0
    lo, hi = wilson_ci(k, n)
    return {
        f"{cond}_acc": float(np.nanmean(epu_vec)) if n else np.nan,
        f"{cond}_acc_ci_low": lo,
        f"{cond}_acc_ci_high": hi,
        f"{cond}_json_valid_rate": g[f"{cond}_json_valid"].mean(),
        f"{cond}_excerpt_in_target_rate": g[f"{cond}_excerpt_in_target"].mean(),
        f"{cond}_confidence_mean": g[f"{cond}_confidence"].mean(),
        f"{cond}_aux_mean": g[f"{cond}_aux_mean"].mean(),
        f"{cond}_task_completion_mean": g[f"{cond}_task_completion_count"].mean(),
        f"{cond}_completion_share_mean": g[f"{cond}_completion_share"].mean(),
        f"{cond}_parse_failure_rate": 1.0 - g[f"{cond}_json_valid"].mean(),
    }

def add_class_specific_metrics(rec: dict, g: pd.DataFrame, cond: str):
    for gold_val in [1, 0]:
        mask = (g["gold_epu"] == gold_val)
        vec = g.loc[mask, f"{cond}_epu_correct"].astype(float).to_numpy()
        rec[f"{cond}_acc_gold{gold_val}"] = float(np.nanmean(vec)) if len(vec) else np.nan
    return rec

pair_rows = []
overall_rows = []

for model_name, g in wide_df.groupby("llm_name", sort=False):
    rec = {"llm_name": model_name, "n_rows": len(g)}
    for cond_meta in CONDITIONS:
        cond = cond_meta["condition"]
        rec.update(summarize_condition(g, cond))
        rec = add_class_specific_metrics(rec, g, cond)

    # all pairwise proof table
    for a_meta, b_meta in combinations(CONDITIONS, 2):
        a = a_meta["condition"]
        b = b_meta["condition"]
        comp = summarize_pair(
            g[f"{a}_epu_correct"],
            g[f"{b}_epu_correct"],
            g[f"{a}_epu_pred"],
            g[f"{b}_epu_pred"],
            f"{a} -> {b}",
        )
        pair_row = {"llm_name": model_name, "cond_a": a, "cond_b": b, **comp}
        pair_rows.append(pair_row)

    first_name = target_condition_name(PRIMARY_BUNDLE, "epu_first")
    last_name = target_condition_name(PRIMARY_BUNDLE, "epu_last")
    comp_single_first = summarize_pair(g["single_focus_epu_correct"], g[f"{first_name}_epu_correct"], g["single_focus_epu_pred"], g[f"{first_name}_epu_pred"], f"single_focus -> {first_name}")
    comp_single_last = summarize_pair(g["single_focus_epu_correct"], g[f"{last_name}_epu_correct"], g["single_focus_epu_pred"], g[f"{last_name}_epu_pred"], f"single_focus -> {last_name}")
    comp_first_last = summarize_pair(g[f"{first_name}_epu_correct"], g[f"{last_name}_epu_correct"], g[f"{first_name}_epu_pred"], g[f"{last_name}_epu_pred"], f"{first_name} -> {last_name}")

    rec["primary_bundle"] = PRIMARY_BUNDLE
    rec["single_to_first_delta_pp"] = comp_single_first["delta_pp"]
    rec["single_to_first_mcnemar_p"] = comp_single_first["mcnemar_p"]
    rec["single_to_first_flip_rate"] = comp_single_first["flip_rate"]

    rec["single_to_last_delta_pp"] = comp_single_last["delta_pp"]
    rec["single_to_last_mcnemar_p"] = comp_single_last["mcnemar_p"]
    rec["single_to_last_flip_rate"] = comp_single_last["flip_rate"]

    rec["first_to_last_delta_pp"] = comp_first_last["delta_pp"]
    rec["first_to_last_delta_ci_low"] = comp_first_last["delta_ci_low"]
    rec["first_to_last_delta_ci_high"] = comp_first_last["delta_ci_high"]
    rec["first_to_last_mcnemar_p"] = comp_first_last["mcnemar_p"]
    rec["first_to_last_flip_rate"] = comp_first_last["flip_rate"]
    rec["failure_mode"] = classify_q3_failure(rec)
    overall_rows.append(rec)

pairwise_df = pd.DataFrame(pair_rows)
pairwise_df.to_csv(OUT_DIR / "q3_pairwise_proof_upgraded.csv", index=False)

overall_df = pd.DataFrame(overall_rows)
overall_df.to_csv(OUT_DIR / "q3_summary_overall_upgraded.csv", index=False)

# ---------------------------
# SUMMARIES BY DISAGREEMENT / GOLD / TRUNCATION
# ---------------------------
def summarize_group(g: pd.DataFrame, group_keys: dict):
    rec = dict(group_keys)
    rec["n_rows"] = len(g)
    for cond_meta in CONDITIONS:
        cond = cond_meta["condition"]
        rec.update(summarize_condition(g, cond))
        rec = add_class_specific_metrics(rec, g, cond)
    first_name = target_condition_name(PRIMARY_BUNDLE, "epu_first")
    last_name = target_condition_name(PRIMARY_BUNDLE, "epu_last")
    comp = summarize_pair(g[f"{first_name}_epu_correct"], g[f"{last_name}_epu_correct"], g[f"{first_name}_epu_pred"], g[f"{last_name}_epu_pred"], f"{first_name} -> {last_name}")
    rec["first_to_last_delta_pp"] = comp["delta_pp"]
    rec["first_to_last_mcnemar_p"] = comp["mcnemar_p"]
    rec["first_to_last_flip_rate"] = comp["flip_rate"]
    return rec

by_dis_rows = []
for (model_name, flag), g in wide_df.groupby(["llm_name", "disagreement_flag"], sort=False):
    by_dis_rows.append(summarize_group(g, {"llm_name": model_name, "disagreement_flag": int(flag)}))
pd.DataFrame(by_dis_rows).to_csv(OUT_DIR / "q3_summary_by_disagreement_upgraded.csv", index=False)

by_gold_rows = []
for (model_name, gold_epu), g in wide_df.groupby(["llm_name", "gold_epu"], sort=False):
    by_gold_rows.append(summarize_group(g, {"llm_name": model_name, "gold_epu": int(gold_epu)}))
pd.DataFrame(by_gold_rows).to_csv(OUT_DIR / "q3_summary_by_gold_epu_upgraded.csv", index=False)

by_trunc_rows = []
for (model_name, trunc_flag), g in wide_df.groupby(["llm_name", "was_truncated"], sort=False):
    by_trunc_rows.append(summarize_group(g, {"llm_name": model_name, "was_truncated": int(trunc_flag)}))
pd.DataFrame(by_trunc_rows).to_csv(OUT_DIR / "q3_summary_by_truncation_upgraded.csv", index=False)

# ---------------------------
# SUBTASK TABLE
# ---------------------------
subtask_rows = []
for model_name, g in wide_df.groupby("llm_name", sort=False):
    for cond_meta in CONDITIONS:
        cond = cond_meta["condition"]
        if cond_meta["kind"] != "multi":
            continue
        for field in ALL_AUX_FIELDS:
            req = int(field in cond_meta["requested_fields"])
            vec = g[f"{cond}_{field}_correct"].astype(float).to_numpy()
            requested_mask = g[f"{cond}_{field}_requested"] == 1
            vec_req = g.loc[requested_mask, f"{cond}_{field}_correct"].astype(float).to_numpy()
            n = int(np.sum(~np.isnan(vec_req)))
            k = int(np.nansum(vec_req)) if n else 0
            lo, hi = wilson_ci(k, n)
            subtask_rows.append({
                "llm_name": model_name,
                "condition": cond,
                "bundle": cond_meta["bundle"],
                "order": cond_meta["order"],
                "subtask": field,
                "requested": req,
                "acc": float(np.nanmean(vec_req)) if n else np.nan,
                "acc_ci_low": lo,
                "acc_ci_high": hi,
                "present_rate": g.loc[requested_mask, f"{cond}_{field}_present"].mean() if requested_mask.any() else np.nan,
            })
pd.DataFrame(subtask_rows).to_csv(OUT_DIR / "q3_subtask_table_upgraded.csv", index=False)

# ---------------------------
# DEPENDENCY TABLE
# ---------------------------
dependency_rows = []
for model_name, g in wide_df.groupby("llm_name", sort=False):
    for cond_meta in CONDITIONS:
        cond = cond_meta["condition"]
        aux = g[f"{cond}_aux_mean"].astype(float)
        main = g[f"{cond}_epu_correct"].astype(float)
        if aux.notna().sum() == 0:
            continue
        high_mask = aux >= 0.6
        low_mask = aux < 0.6
        dependency_rows.append({
            "llm_name": model_name,
            "condition": cond,
            "main_acc_when_aux_high": float(np.nanmean(main[high_mask])) if high_mask.any() else np.nan,
            "main_acc_when_aux_low": float(np.nanmean(main[low_mask])) if low_mask.any() else np.nan,
            "n_aux_high": int(high_mask.sum()),
            "n_aux_low": int(low_mask.sum()),
        })
pd.DataFrame(dependency_rows).to_csv(OUT_DIR / "q3_dependency_table_upgraded.csv", index=False)

# ---------------------------
# HARDEST HARMED ROWS (primary comparison)
# ---------------------------
primary_first = target_condition_name(PRIMARY_BUNDLE, "epu_first")
primary_last = target_condition_name(PRIMARY_BUNDLE, "epu_last")

hard_rows = []
for model_name, g in wide_df.groupby("llm_name", sort=False):
    tmp = g.copy()
    tmp["single_to_last_hurt"] = ((tmp["single_focus_epu_correct"] == 1.0) & (tmp[f"{primary_last}_epu_correct"] == 0.0)).astype(int)
    tmp["first_to_last_hurt"] = ((tmp[f"{primary_first}_epu_correct"] == 1.0) & (tmp[f"{primary_last}_epu_correct"] == 0.0)).astype(int)
    harmed = tmp[(tmp["single_to_last_hurt"] == 1) | (tmp["first_to_last_hurt"] == 1)].copy()
    harmed["llm_name"] = model_name
    keep_cols = [
        "llm_name", "article_key", "gold_epu", "disagreement_flag", "newspaper", "year",
        "single_focus_epu_pred", f"{primary_first}_epu_pred", f"{primary_last}_epu_pred",
        "single_focus_focus_excerpt", f"{primary_first}_focus_excerpt", f"{primary_last}_focus_excerpt",
        "single_to_last_hurt", "first_to_last_hurt",
    ]
    hard_rows.append(harmed[keep_cols])
hard_df = pd.concat(hard_rows, axis=0, ignore_index=True) if hard_rows else pd.DataFrame()
hard_df.to_csv(OUT_DIR / "q3_most_harmed_rows_upgraded.csv", index=False)

# ---------------------------
# ARTICLE x MODEL MATRIX
# ---------------------------
matrix_rows = []
for row in wide_df.to_dict(orient="records"):
    first_name = target_condition_name(PRIMARY_BUNDLE, "epu_first")
    last_name = target_condition_name(PRIMARY_BUNDLE, "epu_last")
    matrix_rows.append({
        "article_key": row["article_key"],
        "llm_name": row["llm_name"],
        "gold_epu": row["gold_epu"],
        "disagreement_flag": row["disagreement_flag"],
        "single_acc": row.get("single_focus_epu_correct"),
        "single_long_control_acc": row.get("single_long_control_epu_correct"),
        "primary_first_acc": row.get(f"{first_name}_epu_correct"),
        "primary_last_acc": row.get(f"{last_name}_epu_correct"),
        "primary_first_pred": row.get(f"{first_name}_epu_pred"),
        "primary_last_pred": row.get(f"{last_name}_epu_pred"),
        "order_flip": int((row.get(f"{first_name}_epu_pred") is not None) and (row.get(f"{primary_last}_epu_pred") is not None) and (row.get(f"{first_name}_epu_pred") != row.get(f"{primary_last}_epu_pred"))),
    })
pd.DataFrame(matrix_rows).to_csv(OUT_DIR / "q3_article_model_matrix_upgraded.csv", index=False)

# ---------------------------
# MANIFEST
# ---------------------------
manifest = pd.DataFrame([
    {"file": "pilot_rows_q3_upgraded.csv", "path": str(pilot_path)},
    {"file": "run_status_q3_upgraded.csv", "path": str(OUT_DIR / "run_status_q3_upgraded.csv")},
    {"file": "q3_results_wide_upgraded.csv", "path": str(wide_path)},
    {"file": "q3_results_long_upgraded.csv", "path": str(long_path)},
    {"file": "q3_summary_overall_upgraded.csv", "path": str(OUT_DIR / "q3_summary_overall_upgraded.csv")},
    {"file": "q3_pairwise_proof_upgraded.csv", "path": str(OUT_DIR / "q3_pairwise_proof_upgraded.csv")},
    {"file": "q3_summary_by_gold_epu_upgraded.csv", "path": str(OUT_DIR / "q3_summary_by_gold_epu_upgraded.csv")},
    {"file": "q3_summary_by_disagreement_upgraded.csv", "path": str(OUT_DIR / "q3_summary_by_disagreement_upgraded.csv")},
    {"file": "q3_summary_by_truncation_upgraded.csv", "path": str(OUT_DIR / "q3_summary_by_truncation_upgraded.csv")},
    {"file": "q3_subtask_table_upgraded.csv", "path": str(OUT_DIR / "q3_subtask_table_upgraded.csv")},
    {"file": "q3_dependency_table_upgraded.csv", "path": str(OUT_DIR / "q3_dependency_table_upgraded.csv")},
    {"file": "q3_most_harmed_rows_upgraded.csv", "path": str(OUT_DIR / "q3_most_harmed_rows_upgraded.csv")},
    {"file": "q3_article_model_matrix_upgraded.csv", "path": str(OUT_DIR / "q3_article_model_matrix_upgraded.csv")},
])
manifest.to_csv(OUT_DIR / "q3_manifest_upgraded.csv", index=False)

print("\nSaved outputs:")
display(manifest)
print("\nOverall summary preview:")
display(pd.read_csv(OUT_DIR / "q3_summary_overall_upgraded.csv"))
